# Checkers AI: Tool Use, Fine-Tuning, and Evaluation

This workshop investigates whether fine-tuning **Qwen2.5-0.5B-Instruct** with LoRA improves its ability to use checkers tools and complete a legal turn. We compare the original model and the trained adapter on the same evaluation sample.

The model proposes tool calls. A **harness** parses those calls, executes supported tools, returns their results, and limits retries. The `pydraughts` engine determines legal moves and updates the board. This lets us measure both communication with the harness and successful move execution.

### Before running

1. Upload the project ZIP as a Kaggle Dataset and attach it using **Add Input**. It must contain `agent/`, `checkers/`, `data/`, and `evaluation/`.
2. Enable a **GPU** and turn **Internet** on for package installation and model downloads. If GPU options are disabled, check account phone verification.
3. Run the installation cell, then configure the paths in the following cell. Run subsequent cells in order.

| Setting | Purpose |
|---|---|
| `PROJECT_ROOT` | Actual attached folder containing the four project directories. Copy its path from Kaggle Input. |
| `BASE_MODEL` | Defaults to `Qwen/Qwen2.5-0.5B-Instruct`; can also point to attached model files. |
| `TRAIN_ADAPTER` | `True` trains an adapter; `False` uses an existing adapter at `ADAPTER_PATH`. |
| `FAST_MODE` | `True` uses the first 256 training trajectories; `False` uses all training trajectories. |
| `OUTPUT_DIR` / `ADAPTER_PATH` | Where the adapter is saved and subsequently loaded. |

For the dataset attached in this session, set the configuration cell to:

```python
PROJECT_ROOT = Path("/kaggle/input/datasets/kishlayabhagat/prism-checkers-agent")
```

Other accounts or dataset names may have different paths. Confirm that setup prints a GPU name and **`Data exists: True`**. The setup error about an outdated dataset can also mean `PROJECT_ROOT` points to the wrong folder.

We will validate the data, check the base model's first action, train and reload an adapter, then evaluate complete turns. Keep the resulting reports to support conclusions about whether fine-tuning helped.


In [ ]:
# Qwen2.5 native tool templates need Transformers 4.42 or newer.
!pip -q install pydraughts==0.6.7 transformers==4.49.0 peft==0.14.0 datasets==3.3.2 accelerate==1.4.0

In [ ]:
import os, sys, json, statistics, torch
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/kaggle/input/datasets/simplynb14/prism-workshop-dataset"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BASE_MODEL = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
ALLOW_HF_DOWNLOAD = os.environ.get("ALLOW_HF_DOWNLOAD", "1") == "1"
OUTPUT_DIR = Path("/kaggle/working/prism-checkers-lora")
ADAPTER_PATH = Path(os.environ.get("ADAPTER_PATH", str(OUTPUT_DIR)))
TRAIN_ADAPTER = True  # Set False to evaluate an attached adapter without training.
FAST_MODE = True      # False uses all training trajectories; both settings train for one epoch.
sys.path.insert(0, str(PROJECT_ROOT))
PROMPTS_FILE = PROJECT_ROOT / "agent" / "prompts.py"
HARNESS_FILE = PROJECT_ROOT / "agent" / "harness.py"
MODEL_FILE = PROJECT_ROOT / "agent" / "model.py"
PROTOCOL_FILE = PROJECT_ROOT / "agent" / "protocol.py"
TOOLS_FILE = PROJECT_ROOT / "agent" / "tools.py"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training is not recommended on CPU)")
print("Project:", PROJECT_ROOT.resolve())
print("Data exists:", (DATA_DIR / "train_trajectories.jsonl").exists())

## 1. Inspect the Game and Dataset

We use **English/American checkers** on an 8 x 8 board. Moves use numbered dark squares from 1 to 32: `9-13` is a quiet move and `x` marks a capture. Captures are mandatory, and a complete sequence of jumps is handled as one turn. The engine supplies the legal moves; the model must submit one through `make_move`.

A **trajectory** is one recorded agent turn: the starting position, conversation, tool calls, tool responses, and the teacher's chosen move. The teacher uses a simple heuristic favoring captures, material gains, and promotion. These examples teach a tool-use procedure and imitation of that heuristic.

| File | Contents and use | Records in this bundle |
|---|---|---:|
| `train_trajectories.jsonl` | Conversations used to train the adapter | 684 |
| `validation_trajectories.jsonl` | Conversations used to measure validation loss | 64 |
| `test_trajectories.jsonl` | Reserved test conversations | 126 |
| `test_positions.jsonl` | Positions used by the evaluation cells, including five curated cases | 131 |

Source games are separated across trajectory splits. Positions can still recur across different games, so this is not a guarantee that every test board is unique to the test set.

**Procedure:** Run the next cell. It checks required trajectory fields, assistant action parsing, teacher move legality, split game IDs, and stored test legal moves. It then displays one training position and its legal moves. Confirm the counts above and that no validation exception occurs before continuing.


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from checkers.engine import CheckersGame
from data.validate_dataset import validate_directory

print("Dataset validation:", validate_directory(DATA_DIR))
example = json.loads((DATA_DIR / "train_trajectories.jsonl").read_text().splitlines()[0])
game = CheckersGame(initial_fen=example["start_fen"])
print("Example player:", example["player"])
print("Legal moves:", game.legal_moves())
print(game.state()["board"])

### Baseline Protocol Check: The First Action

Before training, we ask the base model for **one response on each of the first 20 test positions**. This checks whether it can produce a parseable action and begin with the expected `get_board_state` call. It does not execute a complete turn.

Qwen's chat template supplies the tool definitions and formats a native call like this:

```text
<tool_call>
{"name": "get_board_state", "arguments": {}}
</tool_call>
```

The parser converts that response into the harness's internal representation:

```json
{"action": "call_tool", "tool": "get_board_state", "arguments": {}}
```

The supported tools are `get_board_state`, `get_legal_moves`, `get_move_history`, `make_move`, and `check_game_status`. The usual training sequence is **read the board -> list legal moves -> submit a move**. Tool results return `ok: true` with a result, or `ok: false` with an error.

**Procedure:** Run the next cell and wait for the model download and all 20 responses to finish. Inspect the example response and the final report:

- `valid_output_rate`: fraction of responses accepted by the action parser.
- `first_tool_accuracy`: fraction of parseable responses selecting `get_board_state` in this preliminary check.

A value of `1.0` means 100% for that metric. Even perfect results here do not establish correct tool arguments, legal move completion, or playing strength; those require the later evaluation.


In [ ]:
from pathlib import Path
if not Path(BASE_MODEL).exists() and "/" in BASE_MODEL:
    print("Using Hugging Face model ID. Enable internet for the first run, or attach its files as a Kaggle Model and set BASE_MODEL to that path.")
from transformers import AutoModelForCausalLM, AutoTokenizer
from agent.prompts import SYSTEM_PROMPT, initial_user_message
from agent.protocol import parse_action, ProtocolError
from agent.model import format_qwen_tool_chat
from checkers.engine import CheckersGame

if Path(BASE_MODEL).exists() or ALLOW_HF_DOWNLOAD:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="auto", device_map="auto")
    rows = [json.loads(line) for line in (DATA_DIR / "test_positions.jsonl").read_text().splitlines()[:20]]
    valid, expected_tool = 0, 0
    for index, row in enumerate(rows):
        game = CheckersGame(initial_fen=row["fen"])
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": initial_user_message(game.state(), game.history)},
        ]
        prompt = format_qwen_tool_chat(tokenizer, prompt_messages, add_generation_prompt=True)
        batch = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            output = model.generate(**batch, max_new_tokens=128, do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(output[0, batch["input_ids"].shape[1]:], skip_special_tokens=False).strip()
        if index == 0:
            print("Example baseline response:", repr(raw[:600]))
        try:
            action = parse_action(raw)
            valid += 1
            expected_tool += action.get("tool") == "get_board_state"
        except ProtocolError:
            pass
    print({"valid_output_rate": valid / max(1, len(rows)), "first_tool_accuracy": expected_tool / max(1, valid), "positions": len(rows)})
else:
    print("Attach the Qwen model as a Kaggle input and set BASE_MODEL to its mounted folder, then rerun this cell.")

## 2. Fine-Tune with LoRA

LoRA trains small adapter weights while keeping the base model weights frozen. This cell adds adapters to the attention `q_proj` and `v_proj` layers, using rank 8, alpha 16, and dropout 0.05. The learning rate is `5e-5`, and training runs for one epoch.

Each trajectory becomes several examples, one for each assistant tool call. Earlier messages form the prompt; the next call is the target. **Only target tokens contribute to the training loss.** Board descriptions, prior conversation, and tool responses are masked from the loss. Long examples are truncated from the beginning to fit 2,048 tokens. Stored `finish_turn` actions are excluded from training targets; the harness returns as soon as a legal `make_move` succeeds.

With `FAST_MODE = True`, this bundle supplies **256 trajectories and 768 action examples**, plus **192 validation action examples**. The fast subset uses the first records in file order and may omit curated examples stored later in the file. Set `FAST_MODE = False` to include the full training split.

**Procedure:**

1. Run the next cell and inspect the prompt and target previews to confirm the target is a tool call.
2. Wait for training and validation to finish. Compare training and validation losses, keeping in mind that token prediction loss alone does not measure complete-turn success.
3. Confirm `Saved adapter:` appears. The cell saves adapter weights, tokenizer files, and `training_metadata.json` under `OUTPUT_DIR`.
4. Check the reload result. `Parsed action:` confirms that the saved adapter can be loaded and produces a parseable first action in the starting position.

The reload check is limited to one response. Use the next section to determine whether the adapter completes legal turns more reliably than the base model.


In [ ]:
if TRAIN_ADAPTER:
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
    from peft import LoraConfig, TaskType, PeftModel, get_peft_model
    from agent.protocol import parse_action
    from agent.prompts import SYSTEM_PROMPT, initial_user_message
    from agent.model import format_qwen_tool_chat
    from checkers.engine import CheckersGame
    from torch.nn.utils.rnn import pad_sequence

    print("Validated files:", validate_directory(DATA_DIR))

    def read_jsonl(path):
        return [json.loads(line) for line in Path(path).read_text().splitlines()]

    def qwen_messages(messages):
        converted = []
        for message in messages:
            role, content = message["role"], str(message["content"])
            if role == "system":
                converted.append({"role": "system", "content": SYSTEM_PROMPT})
            elif role == "tool":
                converted.append({
                    "role": "tool",
                    "name": message.get("name", "tool"),
                    "content": content,
                })
            elif role == "assistant":
                try:
                    action = parse_action(content)
                except ValueError:
                    action = None
                if action and action["action"] == "call_tool":
                    converted.append({
                        "role": "assistant",
                        "tool_calls": [{
                            "type": "function",
                            "function": {
                                "name": action["tool"],
                                "arguments": action["arguments"],
                            },
                        }],
                    })
                else:
                    converted.append({"role": "assistant", "content": content})
            else:
                converted.append({"role": role, "content": content})
        return converted

    train_rows = read_jsonl(DATA_DIR / "train_trajectories.jsonl")
    valid_rows = read_jsonl(DATA_DIR / "validation_trajectories.jsonl")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    MAX_SEQ_LENGTH = 2048

    def make_action_examples(rows):
        examples = []
        for row in rows:
            raw_messages = row["messages"]
            for target_index, target in enumerate(raw_messages):
                if target["role"] != "assistant":
                    continue
                try:
                    action = parse_action(target["content"])
                except ValueError:
                    continue
                if action["action"] != "call_tool":
                    continue

                prefix = qwen_messages(raw_messages[:target_index])
                prompt_text = format_qwen_tool_chat(tokenizer, prefix, add_generation_prompt=True)
                full_text = format_qwen_tool_chat(
                    tokenizer, qwen_messages(raw_messages[:target_index + 1]), add_generation_prompt=False
                )
                if not full_text.startswith(prompt_text):
                    raise ValueError("Qwen chat template did not preserve the prompt/completion boundary")

                prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
                input_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
                # Tokenization can merge the final prompt whitespace with the first
                # completion token, so mask their shared token prefix.
                prompt_length = 0
                for prompt_token, full_token in zip(prompt_ids, input_ids):
                    if prompt_token != full_token:
                        break
                    prompt_length += 1
                if prompt_length == len(input_ids):
                    continue

                if len(input_ids) > MAX_SEQ_LENGTH:
                    excess = len(input_ids) - MAX_SEQ_LENGTH
                    input_ids = input_ids[excess:]
                    prompt_length = max(0, prompt_length - excess)
                labels = [-100] * prompt_length + input_ids[prompt_length:]
                examples.append({"input_ids": input_ids, "labels": labels})
        return examples

    train_limit = 256 if FAST_MODE else len(train_rows)
    train_examples = make_action_examples(train_rows[:train_limit])
    valid_examples = make_action_examples(valid_rows)
    train_data = Dataset.from_list(train_examples)
    valid_data = Dataset.from_list(valid_examples)
    print("Training trajectories:", train_limit, "of", len(train_rows))
    print("Assistant tool-action examples:", len(train_data), "validation:", len(valid_data))
    preview = train_data[0]
    first_target = [token for token, label in zip(preview["input_ids"], preview["labels"]) if label != -100]
    print("Masked prompt preview:", tokenizer.decode(preview["input_ids"][:min(300, len(preview["input_ids"]))]))
    print("Target action preview:", tokenizer.decode(first_target))

    def completion_only_collator(features):
        input_tensors = [torch.tensor(item["input_ids"], dtype=torch.long) for item in features]
        label_tensors = [torch.tensor(item["labels"], dtype=torch.long) for item in features]
        input_ids = pad_sequence(input_tensors, batch_first=True, padding_value=tokenizer.pad_token_id)
        labels = pad_sequence(label_tensors, batch_first=True, padding_value=-100)
        attention_mask = torch.zeros_like(input_ids)
        for index, item in enumerate(features):
            attention_mask[index, :len(item["input_ids"])] = 1
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
    )
    model = get_peft_model(base_model, lora)
    model.print_trainable_parameters()
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        warmup_ratio=0.05,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        fp16=True,
        label_names=["labels"],
        seed=7,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=valid_data,
        data_collator=completion_only_collator,
    )
    train_result = trainer.train()
    print(train_result)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    (OUTPUT_DIR / "training_metadata.json").write_text(json.dumps({
        "base_model": BASE_MODEL,
        "dataset": "teacher-generated English draughts tool trajectories",
        "train_trajectories": train_limit,
        "train_action_examples": len(train_data),
        "validation_action_examples": len(valid_data),
        "completion_only_loss": True,
        "qwen_native_tool_calls": True,
        "epochs": 1,
        "fast_mode": FAST_MODE,
        "seed": 7,
    }, indent=2))
    print("Saved adapter:", OUTPUT_DIR)

    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="auto", device_map="auto")
    reloaded = PeftModel.from_pretrained(base, OUTPUT_DIR)
    reloaded.eval()
    smoke_game = CheckersGame()
    smoke_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": initial_user_message(smoke_game.state(), smoke_game.history)},
    ]
    smoke_prompt = format_qwen_tool_chat(tokenizer, smoke_messages, add_generation_prompt=True)
    smoke_batch = tokenizer(smoke_prompt, return_tensors="pt").to(reloaded.device)
    with torch.inference_mode():
        smoke_output = reloaded.generate(
            **smoke_batch, max_new_tokens=128, do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    smoke_raw = tokenizer.decode(smoke_output[0, smoke_batch["input_ids"].shape[1]:], skip_special_tokens=False).strip()
    print("Adapter first action:", repr(smoke_raw[:600]))
    try:
        print("Parsed action:", parse_action(smoke_raw))
    except ValueError as exc:
        print("Protocol check failed:", exc)
else:
    print("Skipping training. Set ADAPTER_PATH to the attached adapter directory for the comparison below.")


# Enable for the organizer's fuller opponent comparison; this adds model-generation time.
RUN_MATCHES = False
if RUN_MATCHES and baseline is not None:
    from evaluation.arena import play_match
    from evaluation.metrics import summarize_matches
    models = {"baseline": baseline}
    if Path(ADAPTER_PATH).exists(): models["fine_tuned"] = finetuned
    match_reports = {}
    match_details = {}
    for label, model in models.items():
        matches = [play_match(model, model_side=side, seed=7) for side in ("black", "red")]
        match_reports[label] = summarize_matches(matches)
        match_details[label] = matches
    saved_report = {
        "positions": {"baseline": base_report, "fine_tuned": adapter_report if Path(ADAPTER_PATH).exists() else None},
        "matches": match_reports,
        "match_details": match_details,
    }
    Path("evaluation_report.json").write_text(json.dumps(saved_report, indent=2))
    print("Win/draw/loss vs heuristic teacher:", json.dumps(match_reports, indent=2))
    print("Saved match moves and summaries to evaluation_report.json")

In [ ]:
from collections import Counter
from agent.model import HuggingFaceAgentModel
from agent.harness import AgentHarness
from checkers.engine import CheckersGame
rows = [json.loads(line) for line in (DATA_DIR / "test_positions.jsonl").read_text().splitlines()]
print("Held-out positions:", len(rows))

def evaluate(model, positions):
    results = []
    failure_reasons = Counter()
    example_invalid_response = None
    for row in positions:
        game = CheckersGame(initial_fen=row["fen"])
        harness = AgentHarness(model=model, game=game, max_tool_calls=6, max_invalid_outputs=1, fallback_enabled=True)
        turn = harness.play_turn()
        proposals = [event for event in harness.events if event["type"] == "proposal"]
        failures = [event for event in harness.events if event["type"] == "validation_failure"]
        calls = [event for event in harness.events if event["type"] == "tool_call"]
        tool_results = [event for event in harness.events if event["type"] == "tool_result"]
        first_tool = calls[0]["tool"] if calls else None
        failure_reasons.update(event["error"] for event in failures)
        if example_invalid_response is None and failures:
            first_proposal = proposals[0]["raw"] if proposals else "<no proposal>"
            example_invalid_response = {
                "error": failures[0]["error"],
                "raw": str(first_proposal)[:600],
            }
        results.append({
            "model_move": turn["move"],
            "legal_before_fallback": turn["source"] == "model" and turn["move"] in row["legal_moves"],
            "task_completed": turn["source"] == "model",
            "teacher_match": turn["source"] == "model" and turn["move"] == row["teacher_move"],
            "valid_output_rate": 1 - len(failures) / max(1, len(proposals)),
            "first_tool_correct": first_tool == "get_board_state",
            "tool_arguments_valid": bool(tool_results) and all(event["ok"] for event in tool_results),
            "fallback_rate": turn["source"] == "fallback",
            "latency_ms": statistics.mean([p["latency_ms"] for p in proposals]) if proposals else None,
        })
    return {
        "positions": len(results),
        "legal_move_rate_before_fallback": sum(r["legal_before_fallback"] for r in results) / max(1, len(results)),
        "task_completion_rate": statistics.mean(r["task_completed"] for r in results),
        "tool_argument_validity": statistics.mean(r["tool_arguments_valid"] for r in results),
        "fallback_rate": statistics.mean(r["fallback_rate"] for r in results),
        "teacher_move_agreement": sum(r["teacher_match"] for r in results) / max(1, len(results)),
        "valid_output_rate": statistics.mean(r["valid_output_rate"] for r in results),
        "first_tool_accuracy": statistics.mean(r["first_tool_correct"] for r in results),
        "mean_latency_ms": statistics.mean([r["latency_ms"] for r in results if r["latency_ms"] is not None]) if any(r["latency_ms"] is not None for r in results) else None,
        "protocol_failure_reasons": dict(failure_reasons),
        "example_invalid_response": example_invalid_response,
    }

sample = rows[: min(35, len(rows))] + [row for row in rows if row.get("scenario")]  # Include every curated edge case.
baseline = None
if Path(BASE_MODEL).exists() or ALLOW_HF_DOWNLOAD:
    baseline = HuggingFaceAgentModel(model_name=BASE_MODEL, adapter_path=None)
    base_report = evaluate(baseline, sample)
    print("Baseline:", json.dumps(base_report, indent=2))
else:
    print("Attach Qwen as a Kaggle Model input or enable internet/HF access, then set BASE_MODEL.")

In [ ]:
if baseline is not None and Path(ADAPTER_PATH).exists():
    finetuned = HuggingFaceAgentModel(model_name=BASE_MODEL, adapter_path=str(ADAPTER_PATH))
    adapter_report = evaluate(finetuned, sample)
    report = {"baseline": base_report, "fine_tuned": adapter_report, "interpretation": "This report measures the sampled positions only; teacher agreement is not strategic strength."}
    Path("evaluation_report.json").write_text(json.dumps(report, indent=2))
    print("Fine-tuned:", json.dumps(adapter_report, indent=2))
    print("Saved evaluation_report.json")
elif baseline is not None:
    print("Adapter input not found. Attach the saved LoRA output and set ADAPTER_PATH.")

In [ ]:
# Enable for the organizer's fuller opponent comparison; this adds model-generation time.
RUN_MATCHES = True
if RUN_MATCHES and baseline is not None:
    from evaluation.arena import play_match
    from evaluation.metrics import summarize_matches
    #models = {"baseline": baseline}
    if Path(ADAPTER_PATH).exists(): models["fine_tuned"] = finetuned
    match_reports = {}
    for label, model in models.items():
        matches = [play_match(model, model_side=side, seed=7) for side in ("black", "red")]
        match_reports[label] = summarize_matches(matches)
    #Path("evaluation_report.json").write_text(json.dumps({"positions": {"baseline": base_report, "fine_tuned": adapter_report if Path(ADAPTER_PATH).exists() else None}, "matches": match_reports}, indent=2))
    Path("evaluation_report.json").write_text(json.dumps({"positions": {"fine_tuned": adapter_report if Path(ADAPTER_PATH).exists() else None}, "matches": match_reports}, indent=2))
    print("Win/draw/loss vs heuristic teacher:", json.dumps(match_reports, indent=2))

### Replay a Saved Match

After running the optional match cell, use this cell to replay a saved game inline. It reads the match details from the current notebook session or from `evaluation_report.json`. Set `REPLAY_MODEL` and `REPLAY_MATCH_INDEX` to choose the model and color-side game. The animation has play, pause, and frame controls.

If you restarted the Kaggle session, upload the saved `evaluation_report.json` as a notebook input or rerun the optional match cell first.


In [ ]:
from pathlib import Path
import json
import re
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Choose "baseline" or "fine_tuned" and match index 0 (black) or 1 (red).
REPLAY_MODEL = "fine_tuned"
REPLAY_MATCH_INDEX = 0

if "match_details" in globals():
    saved_matches = match_details
else:
    report_path = Path("evaluation_report.json")
    if not report_path.exists():
        raise FileNotFoundError("Run the optional match cell first, or place evaluation_report.json in the working directory.")
    saved_matches = json.loads(report_path.read_text()).get("match_details", {})

if REPLAY_MODEL not in saved_matches:
    raise KeyError(f"No saved match data for {REPLAY_MODEL!r}. Available: {list(saved_matches)}")
if not 0 <= REPLAY_MATCH_INDEX < len(saved_matches[REPLAY_MODEL]):
    raise IndexError(f"Match index must be 0..{len(saved_matches[REPLAY_MODEL]) - 1}")

match = saved_matches[REPLAY_MODEL][REPLAY_MATCH_INDEX]
replay = CheckersGame()
fens = [replay.board.fen]
for pdn_move in match["moves"]:
    replay.make_move(pdn_move)
    fens.append(replay.board.fen)

def draw_position(ax, fen, frame_index):
    ax.clear()
    ax.set_xlim(0, 8)
    ax.set_ylim(8, 0)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for row in range(8):
        for col in range(8):
            color = "#769656" if (row + col) % 2 else "#eeeed2"
            ax.add_patch(plt.Rectangle((col, row), 1, 1, facecolor=color, edgecolor="none"))
    fields = fen.split(":")
    side_to_move = "Black" if fields[0].startswith("B") else "Red"
    for field in fields[1:]:
        color_code = field[0]
        color = "#30343b" if color_code == "B" else "#c83f49"
        for token in field[1:].split(","):
            if not token:
                continue
            king = token.startswith("K")
            square = int(token[1:] if king else token)
            row = (square - 1) // 4
            offset = (square - 1) % 4
            col = offset * 2 + (1 if row % 2 == 0 else 0)
            ax.add_patch(plt.Circle((col + 0.5, row + 0.5), 0.36, facecolor=color, edgecolor="white", linewidth=1.5))
            if king:
                ax.text(col + 0.5, row + 0.51, "K", ha="center", va="center", color="white", weight="bold", fontsize=14)
    move_text = "Start position" if frame_index == 0 else match["moves"][frame_index - 1]
    ax.set_title(f"{REPLAY_MODEL.replace('_', ' ').title()} vs heuristic · {match['model_side']} · ply {frame_index}/{len(fens)-1} · {side_to_move} to move\n{move_text}", fontsize=11)

fig, ax = plt.subplots(figsize=(6, 6))
def update(frame_index):
    draw_position(ax, fens[frame_index], frame_index)
    return []

animation = FuncAnimation(fig, update, frames=len(fens), interval=650, repeat=False, cache_frame_data=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))